# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook demonstrates loading and exploring a mental health survey dataset using the `mlcroissant` library. All dataset entities are referenced by their Croissant `@id` for clarity and reproducibility.

### Dataset Source
The dataset is modeled with a Croissant schema, accessible from:

https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # The metadata object
# Print dataset name and description (direct property access, not dict subscripting)
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

List the dataset's record sets, fields, and columns by their `@id`s.

In [ ]:
# Examine available record sets, their @ids, and fields
record_set_objs = list(metadata.record_sets)
if not record_set_objs:
    print("No record sets defined in dataset metadata.")
else:
    print(f"{len(record_set_objs)} record set(s) found.")
    for rs in record_set_objs:
        print(f"Record Set @id: {rs.id}")
        # Fields/Columns
        column_ids = [col.id for col in getattr(rs, 'columns', [])]
        print(f"  Columns (@id): {column_ids}")
        # List also human-readable names
        colnames = [col.name for col in getattr(rs, 'columns', [])]
        print(f"  Column Names: {colnames}\n")

## 3. Data Extraction
Load all data from each record set. All variable references use the corresponding `@id` as shown above.

In [ ]:
dataframes = {}
# Gather all record set @ids
record_sets_ids = [rs.id for rs in metadata.record_sets]
print(f"Extracting data from record sets: {record_sets_ids}")
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    # Note: Each record is a dict keyed by column @id
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded DataFrame for record set {record_set_id} with {len(records)} rows.")
    print(f"  Columns (@id): {list(dataframes[record_set_id].columns)}\n")
if record_sets_ids:
    first_rs = record_sets_ids[0]
    print('Sample:', dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. When filtering, transforming, or grouping, always refer to fields using their `@id`.

In [ ]:
# For demonstration, use the first record set and an example numeric field
if record_sets_ids:
    record_set_id = record_sets_ids[0]
    df = dataframes[record_set_id]
    # Find first numeric field (schema:Float, schema:Integer, or schema:Number)
    record_set_obj = next(rs for rs in metadata.record_sets if rs.id == record_set_id)
    numeric_fields = [
        col for col in getattr(record_set_obj, 'columns', [])
        if getattr(col, 'data_type', None) in ['schema:Float','schema:Integer','schema:Number']
    ]
    if numeric_fields:
        numeric_field_obj = numeric_fields[0]
        numeric_field_id = numeric_field_obj.id
        print(f"Using numeric field '@id': {numeric_field_id} (name: {numeric_field_obj.name})")
        # Remove non-numeric/coercible values for processing
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        # Example threshold - median
        threshold = df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Attempt to group by first categorical field
        categorical_fields = [
            col for col in getattr(record_set_obj, 'columns', [])
            if getattr(col, 'data_type', None) == 'schema:Text' and col.id != numeric_field_id
        ]
        if categorical_fields:
            group_field_id = categorical_fields[0].id
            if group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
                print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
                print(grouped_df.head())
    else:
        print("No numeric fields detected in this record set.\n")

## 5. Visualization
Visualizing data distributions. Example below shows a histogram for the first numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets_ids and numeric_fields:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.xlabel(f"{numeric_field_id}")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² Kilifi Mental Health Survey dataset using `mlcroissant`, listing all record set and field `@id`s for clarity. We extracted survey data, performed basic numeric filtering, normalization, and visualizations based strictly on Croissant object references. Use these patterns to further analyze and process AI-ready datasets curated with the FAIR principles in mind.

_Notebook generated for: [FAIR² Kilifi Mental Health Dataset](https://sen.science/doi/10.71728/senscience.vcs2-05nj)_